# MCP Connection: Homeric Greek Passage Workflow

This notebook demonstrates the `perseus` MCP server from Python with **real Greek text data** from Perseus CTS. It uses FastMCP's in-process client transport so the notebook can call the same MCP tools without first starting a separate server process.

You need to do the following:

1. connect to the local MCP server object;
2. inspect available tools;
3. discover Homeric resources;
4. fetch Greek text from *Iliad* 1.1-1.5;
5. do a tiny, transparent word-frequency check on the returned Greek text.

> Requirements: run from the repository root (or keep the path setup cell unchanged), install dependencies with `uv sync` or `pip install -e .`, and have internet access to the upstream Perseus CTS endpoint.


In [3]:
%pip install --quiet fastmcp

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import importlib
import json
import re
import sys

# Make `import server` work when the notebook is opened from examples/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from fastmcp import Client
import server

# Reload local edits when this notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp


## Helper functions

`client.call_tool(...)` returns MCP content blocks. For these text-returning tools, the helper below extracts the text payload. `pretty_json(...)` is just for readable display of JSON responses such as `get_author_resources`.


In [5]:
def tool_text(result):
    """Extract text blocks from a FastMCP CallToolResult."""
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


def pretty_json(text):
    return json.dumps(json.loads(text), ensure_ascii=False, indent=2)


## Connect to the MCP server and list tools

This is an actual MCP client connection. The transport is in-process, but the calls still go through FastMCP's tool interface rather than calling the Python functions directly.


In [6]:
async with Client(mcp) as client:
    tools = await client.list_tools()

[(tool.name, tool.description.splitlines()[0]) for tool in tools]


[('get_passage', 'Get the text of a specific passage using a CTS URN.'),
 ('get_passage_plus',
  'Get passage text plus surrounding metadata/context for a CTS URN.'),
 ('get_passage_plaintext',
  'Get a passage as plain readable text instead of raw CTS XML.'),
 ('get_valid_references',
  'Get valid citations/references for a work, useful for navigation.'),
 ('get_capabilities',
  'Get the list of available texts and editions from Perseus CTS.'),
 ('list_text_groups',
  'List authors/textgroups and their works from CTS capabilities.'),
 ('get_author_resources',
  'List CTS works/editions/translations for an author name or textgroup URN.'),
 ('get_work_resources',
  'List editions/translations/resources for a matching work URN or title.'),
 ('get_label',
  'Get human-readable labels/metadata for a CTS URN (work or edition).'),
 ('get_first_urn',
  'Get the first available reference URN for a work/edition URN.'),
 ('get_prev_next_urn', 'Get previous and next URNs for a passage URN.'),
 ('

## Discover Homeric Greek resources

The `get_author_resources` tool fetches Perseus CTS capabilities and filters them down to Homer's textgroup URN. Using the URN avoids also matching resources such as *Homeric Hymns*.


In [7]:
HOMER_TEXTGROUP = "urn:cts:greekLit:tlg0012"

async with Client(mcp) as client:
    homer = await client.call_tool(
        "get_author_resources",
        {"author": HOMER_TEXTGROUP, "language": "greek"},
    )

homer_json = json.loads(tool_text(homer))
print(pretty_json(tool_text(homer))[:4000])


{
  "query": "urn:cts:greekLit:tlg0012",
  "language": "grc",
  "match_count": 1,
  "authors": [
    {
      "urn": "urn:cts:greekLit:tlg0012",
      "names": [
        "Homer"
      ],
      "works_count": 2,
      "works": [
        {
          "urn": "urn:cts:greekLit:tlg0012.tlg001",
          "language": "grc",
          "titles": [
            "Iliad"
          ],
          "editions": [
            {
              "type": "edition",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
            }
          ],
          "translations": [
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
              "language": "eng",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,38101377, Perseus:bib:isbn,0674991

## Fetch actual Greek text from *Iliad* 1.1-1.5

The code below selects the first advertised Greek *Iliad* edition from the Homer discovery result, then requests its opening five lines.


In [8]:
if not homer_json["authors"]:
    raise RuntimeError(
        "No Homer resources were returned. Rerun the setup and discovery cells "
        "so the notebook reloads the current local server.py."
    )

homer_author = homer_json["authors"][0]
iliad_work = next(
    (work for work in homer_author["works"] if "Iliad" in work["titles"]),
    None,
)
if iliad_work is None or not iliad_work["editions"]:
    raise RuntimeError("No Greek Iliad edition is advertised by the CTS inventory.")

ILIAD_EDITION = iliad_work["editions"][0]["urn"]
ILIAD_1_1_TO_1_5 = f"{ILIAD_EDITION}:1.1-1.5"

async with Client(mcp) as client:
    passage = await client.call_tool(
        "get_passage_plaintext",
        {"urn": ILIAD_1_1_TO_1_5},
    )

greek_text = tool_text(passage)
print(greek_text)


οἰωνοῖσί τε πᾶσι, Διὸς δ᾽ ἐτελείετο βουλή,


## A tiny analysis on the returned Greek

Because `get_passage_plaintext` returns readable Unicode Greek, you can immediately process it with ordinary Python. The simple tokenizer below keeps Greek code points and reports repeated forms in the passage.


In [9]:
tokens = re.findall(r"[Ͱ-Ͽἀ-῿]+", greek_text.lower())
counts = {}
for token in tokens:
    counts[token] = counts.get(token, 0) + 1

print(f"Greek tokens: {len(tokens)}")
print("Most frequent forms:")
for token, count in sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:10]:
    print(f"{token}	{count}")


Greek tokens: 7
Most frequent forms:
βουλή	1
διὸς	1
δ᾽	1
οἰωνοῖσί	1
πᾶσι	1
τε	1
ἐτελείετο	1


## Next steps

Try changing the passage component on the dynamically discovered `ILIAD_EDITION`, for example `f"{ILIAD_EDITION}:1.6-1.10"`.

For the *Odyssey*, select its advertised edition from `homer_json` before constructing a passage URN. Do not assume that an edition suffix remains stable across the live CTS inventory.

Use the companion search/navigation notebook to discover citations and move between passages.


# Notebook version

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 3, 2026</td>
    </tr>
  </table>
</div>